In [ ]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine, text


# database connections 


In [ ]:
with open('../../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['OLTP']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)


# Extract / build
Franjas horarias de negocio (no se requieren minutos exactos).


In [ ]:
def franja_horaria(hora):
    if 0 <= hora <= 5:
        return "Madrugada (00-05)"
    if 6 <= hora <= 11:
        return "Manana (06-11)"
    if 12 <= hora <= 17:
        return "Tarde (12-17)"
    return "Noche (18-23)"

def periodo_dia(hora):
    if 5 <= hora < 12:
        return "AM"
    if 12 <= hora < 19:
        return "PM"
    return "Nocturno"

dim_tiempo = pd.DataFrame({"hora": list(range(24))})
dim_tiempo["franja_horaria"] = dim_tiempo["hora"].apply(franja_horaria)
dim_tiempo["periodo_dia"] = dim_tiempo["hora"].apply(periodo_dia)
dim_tiempo["saved"] = date.today()
dim_tiempo


# load


In [ ]:
with etl_conn.begin() as conn:
    conn.execute(text('Delete from dim_tiempo'))
dim_tiempo.to_sql('dim_tiempo', etl_conn, if_exists='append', index=False)
